## Whole-slide image inference example

This notebook illustrates how to encode tiles from a whole-slide image using a Triton inference server.

If running this notebook on the same machine as the Triton server, prevent TensorFlow from allocating GPU resources.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
import tensorflow as tf
assert len(tf.config.list_physical_devices("GPU")) == 0

In [ ]:
!pip install -e /home/lac5440/simple_triton histomics_stream 'large_image[tiff]'

%load_ext autoreload
%autoreload 2

## Download example data

Download an example whole-slide image and mask pair.

In [ ]:
import pooch

# download whole slide image and corresponding mask
wsi_path = pooch.retrieve(
    fname="TCGA-AN-A0G0-01Z-00-DX1.svs",
    url="https://drive.usercontent.google.com/download?id=19agE_0cWY582szhOVxp9h3kozRfB4CvV&export=download&confirm=t",
    known_hash="d046f952759ff6987374786768fc588740eef1e54e4e295a684f3bd356c8528f",
    path=str(pooch.os_cache("pooch")) + os.sep + "wsi",
)
mask_path = pooch.retrieve(
    fname="TCGA-AN-A0G0-01Z-00-DX1.mask.png",
    url="https://drive.usercontent.google.com/download?id=17GOOHbL8Bo3933rdIui82akr7stbRfta&export=download&confirm=t",
    known_hash="bb657ead9fd3b8284db6ecc1ca8a1efa57a0e9fd73d2ea63ce6053fbd3d65171",
    path=str(pooch.os_cache("pooch")) + os.sep + "wsi",
)

## Create an encoder model

`tf_encoder` creates encoder models using `tensorflow.keras.applications` with configurable input shapes ant types. Selecting a `uint8` dtype allows the encoder model to receive 8-bit inputs instead of float inputs to minimize host-device data transfer. The encoder model is saved into the designated triton server model respository `~/models/` that should be mounted by the triton server container.

In [ ]:
from simple_triton.encoders import tf_encoder

# model parameters
keras_name = "EfficientNetV2S"
model_name = f"{keras_name}.tensorflow"
tile = 224
repository = os.path.join(os.environ["HOME"], "models")
if not os.path.isdir(repository):
    os.mkdir(repository)

# create the model and capture output dimensionality
if not os.path.exists(os.path.join(repository, model_name)):
    dimension_output = tf_encoder(
        repository,
        keras_name,
        model_name,
        input_shape=(tile, tile, 3),
        dtype=tf.uint8,
        pooling="avg",
    )

## Load a model with a minimal configuration

Models served on triton can be configured to control batching, computer resources, and inference optimizations. Models can be configured by providing a configuration during loading with the `TritonModel` class, or by placing a configuration file in the root model directory.

Each triton backend (TensorFlow, ONNX, and python) has unique configuration options and so a configuration class is provided for each. Here we use the `TensorflowConfiguration` class to create a basic configuration. Launching triton server with the `--strict-model-config=false` option enables loading models with a minimal configuration.

In [ ]:
from pprint import pprint
from simple_triton.config import TensorflowConfig

# build a basic configuration specifying only maximum batch size and model name
max_batch_size = 64
name = "EfficientNetV2S.tensorflow"
basic = TensorflowConfig(name, max_batch_size)
pprint(basic.json())

Triton server populates additional fields including information about the input and output dimensions and types and basic optimization for page locking memory used in data transfer.

In [ ]:
from simple_triton.model import TritonModel

# load tensorflow model - set maximum batch size
model = TritonModel(model_name, "localhost:8001")
model.load(config=basic.json())
assert model.is_loaded()
pprint(model.get_config())

## Advanced configuration

Triton can host multiple copies of a model on each GPU using CUDA streams with the `count` option for `InstanceGroup`. This instance group also configures the number of allocated GPUs as well as CPU options. 

Models served with the Tensorflow backend can also benefit from optimizations including mixed precision, XLA compilation, and TensorRT. Mixed precision and TensorRT cannot be used concurrently. The first inference with XLA is slow but subsequent inferences are accelerated.

In [ ]:
from simple_triton.config import (
    InstanceGroup,
    TensorflowMixedPrecision,
    TensorflowOptimization,
)

# create a configuration with two model instances per GPU and mixed precision enabled
instances = InstanceGroup(count=2)
optimization = TensorflowOptimization(
    amp=TensorflowMixedPrecision(),
)
amp_config = TensorflowConfig(
    name, max_batch_size, instance_group=instances, optimization=optimization
)
model.load(config=amp_config.json())
assert model.is_loaded()
pprint(model.get_config())

## Run the inference

First, a histomics stream study is created defining the tiles that need to be read based on the whole-slide image, tissue mask, and desired magnification, tile size, and tile overlap. The chunk parameter is used to group tiles during disk reads to maximize throughput. This study initializes a `LargeimagePrefetch` iterator that generates batches of tiles and tile metadata using prefetching.

This iterator is passed to the inference function that is parameterized by the number of tiles per batch, the number of workers, and the maximum number of pending inferences per worker.

In [ ]:
import numpy as np
from simple_triton.feature_extraction import inference, study
from simple_triton.tile_iterators import TiffPrefetch
from simple_triton.utils import analyze
from time import time

# slide parameters
batch = 64
magnification = 20.0
chunk = 896
mask_threshold = 0.5

# tile iterator parameters
prefetch = 4
workers = 16  # total number of tile
icc = True  # apply ICC color correction

# create a histomics-stream study from a wsi/mask pair
hs_study = study(
    (wsi_path, mask_path),
    t=(tile, tile),
    chunk=(chunk, chunk),
    objective=magnification,
    mask_threshold=mask_threshold,
)

# inference parameters
limit = 1  # limit on number of pending requests per worker
verbose = True  # display inference statistics and debugging information

# start timer
start = time()

# create tile iterator
iterator = TiffPrefetch(hs_study, np.uint8, icc, batch, prefetch, workers)

# inference
features, metadata, times, failures = inference(
    iterator, model_name, url="localhost:8001", limit=limit, rest=0.0
)

# display elapsed time
print(f"Total elapsed time: {time()-start}")

# analyze performance
analyze(times)

## Write features to .tfr

Features can be serialized to TensorFlow record format along with slide and tile metadata.

In [ ]:
from simple_triton.io.tfr_reader import read_record, peek
from simple_triton.io.tfr_writer import write_record

# concatenate features
features = np.concatenate(features[0], axis=0)

# create dummy labels
labels = {"labels": np.random.uniform(size=(10))}

# write to tfrecord
write_record(
    "./example.tfr", features, metadata, labels, structured=False, precision=tf.float16
)

# get list of .tfr variables for de-serialization
serialized = list(tf.data.TFRecordDataset(["./triton.tfr"]))[0]
variables = peek(serialized)

# verify reading
read_record(serialized, variables, structured=False, precision=tf.float16)